In [22]:
import jax 
import jax.numpy as jnp

from probjax.nn.attention import dense_dot_product_attention, memory_efficient_dot_product_attention
from functools import partial

In [2]:
jax.devices()

[cuda(id=0)]

In [23]:
@partial(jax.jit, static_argnums=(3, 4))
def efficient_masked_dot_product_attention(
    query_heads,  # [...,T', H, K]
    key_heads,  # [...,T', H, K]
    value_heads,  # [T, H, V]
    mask, # [T', T]
    return_attention_weights: bool = False,
):
    *leading_dims, sequence_length, _, dim = query_heads.shape
    indices1, indices2 = jnp.nonzero(mask)
    query_heads = jnp.take(
        query_heads, indices1, axis=-3
    )  # [..., E, H, K] Where E is the number of edges
    key_heads = jnp.take(key_heads, indices2, axis=-3)  # [..., E, H, K]
    value_heads = jnp.take(value_heads, indices2, axis=-3)  # [..., E, H, V]
    print(query_heads.shape, key_heads.shape, value_heads.shape)    

    # Attention logits
    attention_logits = jnp.einsum(
        "...ehd,...ehd->...eh", query_heads, key_heads
    ) / jnp.sqrt(dim).astype(key_heads.dtype)
    attention_logits = attention_logits - jnp.max(
        attention_logits, axis=-2, keepdims=True
    )
    attention_weight = jnp.exp(attention_logits)
    attention_normalizer = jax.ops.segment_sum(
        attention_weight,
        indices1,
        num_segments=sequence_length,
        indices_are_sorted=True,
    )
    attention_normalizer = jnp.take(attention_normalizer, indices1, axis=-2)
    attention_weight = attention_weight / attention_normalizer  # [..., eh]

    # Attention weighted values
    attn = attention_weight[..., None] * value_heads
    attn = jax.ops.segment_sum(
        attn, indices1, num_segments=sequence_length, indices_are_sorted=True
    )
    attn = jnp.reshape(attn, (*leading_dims, sequence_length, -1))  # [T', H*V]

    if return_attention_weights:
        return attn, attention_weight
    else:
        return attn, None

In [24]:

mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.1, shape=(1,1, 100, 100)).astype(bool)

q = jax.random.normal(jax.random.PRNGKey(0), shape=(1, 100,8, 10))
k = jax.random.normal(jax.random.PRNGKey(0), shape=(1, 100,8, 10))
v = jax.random.normal(jax.random.PRNGKey(0), shape=(1, 100,8, 10))

In [29]:

efficient_masked_dot_product_attention(q[0],k[0],v[0],mask[0,0])

ValueError: Non-hashable static arguments are not supported. An error occurred during a call to 'efficient_masked_dot_product_attention' while trying to hash an object of type <class 'jaxlib.xla_extension.ArrayImpl'>, [[False False  True ... False  True False]
 [False False False ... False False False]
 [False False False ... False False False]
 ...
 [ True False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]]. The error was:
TypeError: unhashable type: 'ArrayImpl'


In [31]:
@jax.jit
def f1():
    return dense_dot_product_attention(q, k, v, key_size=k.shape[-1])[0]


In [32]:
@jax.jit
def f2():
    return memory_efficient_dot_product_attention(q, k, v)

In [36]:
out1 = f1()

In [37]:
out2 = f2()

In [38]:
jnp.allclose(out1, out2, atol=1e-5)

Array(False, dtype=bool)

In [39]:
%%timeit
f1()

34.7 ms ± 294 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [40]:
%%timeit
f2()

57.9 ms ± 23.8 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
with jax.profiler.trace("/tmp/tensorboard"):
  # Run the operations to be profiled
  y = f1()
  y.block_until_ready()

In [23]:
with jax.profiler.trace("/tmp/tensorboard"):
  # Run the operations to be profiled
  y = f2()
  y.block_until_ready()